# **MongoDB**

A **MongoDB egy nyílt forráskódú NoSQL adatbázis**, amely rugalmas, **JSON-szerű dokumentumokban** (BSON formátumban) tárolja az adatokat.

- Hatékonyan kezeli a nagyméretű, strukturálatlan vagy félig strukturált adatokat
- Könnyen alkalmazkodik a változó adatmodellekhez
- Támogatja a **gyors olvasási/írási műveleteket** valós idejű alkalmazásokhoz és API-khoz
- Zökkenőmentesen **integrálható Pythonnal** olyan könyvtárak használatával, mint a pymongo

### Előfeltételek
1. Helyileg telepített MongoDB (alapértelmezett port: `27017`) vagy MongoDB Atlas (felhő) fiók.
2. JSON dokumentumok alapvető ismerete.

**RDBMS** - Relational Database Management System - relációsadatbázis-kezelő rendszer (Pl. MySQL, PostgreSQL, Oracle)

RDBMS VS MongoDB 

Table - Collection
Row - JSON Document
Index - Index
Join - Embedding  & Linking
Partition - Shard

In [ ]:
# Telepítés
#pip install pymongo


[notice] A new release of pip is available: 25.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## Csatlakozás a MongoDB-hez

A MongoDB-hez való kapcsolódáshoz a `MongoClient`-et használjuk. Az alábbiakban beállítunk egy biztonságos kapcsolatot a helyi szerverhez, és ellenőrizzük azt egy `ping` paranccsal.

>  Az adatbázisnevek **nem tartalmazhatnak kötőjeleket (-)**. Az olyan nevek, mint a *my-Table*, hibát okoznak. Használj helyette aláhúzást (_).

In [ ]:
from pymongo import MongoClient

try:
    client = MongoClient("mongodb://localhost:27017/", serverSelectionTimeoutMS=5000)
    
    client.admin.command("ping")
    print("Connected successfully!")
    
    # Adatbázis és Collection elérése/létrehozása
    mydb = client["my_database"]
    mycollection = mydb["myTable"]
    
except Exception as e:
    print(f"Connection failed: {e}")


Connected successfully!


>  **MongoDB Atlas:** Ha helyi szerver helyett a felhőhöz csatlakoznál, a kliens létrehozását cseréld le a saját kapcsolati karakterláncodra (connection string):
> `client = MongoClient("mongodb+srv://<felhasználónév>:<jelszó>@cluster0.mongodb.net/test?retryWrites=true&w=majority")`

## Adat beszúrása (Create)

A MongoDB-ben az adatokat **szótárként** (dictionary) definiáljuk. Ha nem adunk meg egyedi `_id` kulcsot, a MongoDB automatikusan generál egy 12 bájtos hexadecimális azonosítót.
Adatbeszúrásra az `insert_one()` vagy `insert_many()` metódusokat használjuk.



In [14]:
# Új rekord definiálása
record = {
    "title": "MongoDB and Python",
    "description": "MongoDB is no SQL database",
    "tags": ["mongodb", "database", "NoSQL"],
    "viewers": 104
}

rec = mycollection.insert_one(record)
print(f"Document successfully inserted! Generated ID: {rec.inserted_id}")

Document successfully inserted! Generated ID: 6ad66d8a24e6fe61eb2b30ed


## Adatok lekérdezése (Read)

Az adatok keresésére és szűrésére a következő főbb függvényeket használjuk:
* `find()`: Egy adott lekérdezésnek megfelelő **összes** dokumentum lekérése.
* `find_one()`: Az első megfelelő dokumentum visszaadása.
* `count_documents()`: A feltételnek megfelelő dokumentumok számának lekérése.

In [ ]:
print("--- Search by title ---")
for i in mycollection.find({"title": "MongoDB and Python"}):
    print(i['_id'], i.get('title'), i.get('description'), sep=" - ")

count = mycollection.count_documents({"title": "MongoDB and Python"})
print(f"\nFounded docs: {count}")

print("\n--- All items of the collection ---")
for record in mycollection.find():
    print(record)

--- Search by title ---
6ad66d8a24e6fe61eb2b30ed - MongoDB and Python - MongoDB is no SQL database

Founded docs: 1

--- All items of the collection ---
{'_id': ObjectId('6ad66d8a24e6fe61eb2b30ed'), 'title': 'MongoDB and Python', 'description': 'MongoDB is no SQL database', 'tags': ['mongodb', 'database', 'NoSQL'], 'viewers': 104}


## Adatok frissítése (Update)

A dokumentumok módosítására az `update_one()` (egyetlen dokumentum) vagy az `update_many()` (összes feltételnek megfelelő dokumentum) metódust használjuk.

A frissítés során meg kell adnunk:
1. Egy **szűrőt** (melyik dokumentumot keressük).
2. A **módosítási utasítást** (mit változtatunk meg), amihez a leggyakrabban a `$set` operátort használjuk.


In [33]:
searched = {"title": "MongoDB and Python"}
new_value = {"$set": {"viewers": 150}}

result = mycollection.update_one(searched, new_value)

print(f"Number of modified documents: {result.modified_count}")

updated_doc = mycollection.find_one({"title": "MongoDB and Python"})
print("\nThe updated record:")
print(updated_doc)


Number of modified documents: 1

The updated record:
{'_id': ObjectId('6ad66d8a24e6fe61eb2b30ed'), 'title': 'MongoDB and Python', 'description': 'MongoDB is no SQL database', 'tags': ['mongodb', 'database', 'NoSQL'], 'viewers': 150}


## Több adat egyidejű beszúrása (Tömeges Create)

Ha egyszerre több rekordot szeretnénk feltölteni az adatbázisba, akkor a Python **szótárak listáját (list of dictionaries)** adjuk át az `insert_many()` függvénynek. 

Ez sokkal hatékonyabb, mint egyenként meghívni az `insert_one()`-t, mivel kevesebb hálózati kéréssel jár. A MongoDB ekkor is automatikusan kiosztja az egyedi `_id` azonosítókat minden egyes dokumentumnak.


In [35]:

new_books = [
    {
        "title": "Advanced Python Tips",
        "description": "Mastering decorators and generators",
        "tags": ["python", "advanced", "coding"],
        "viewers": 250
    },
    {
        "title": "Flask API Development",
        "description": "Building microservices with Python",
        "tags": ["python", "flask", "api"],
        "viewers": 180
    },
    {
        "title": "Database Comparison",
        "description": "SQL vs NoSQL performance benchmarks",
        "tags": ["database", "nosql", "sql"],
        "viewers": 95
    }
]

# Tömeges beszúrás a gyűjteménybe
bulk_result = mycollection.insert_many(new_books)

print(f"Successfully inserted {len(bulk_result.inserted_ids)} new documents!")
print("Generated ObjectIDs:")
for index, inserted_id in enumerate(bulk_result.inserted_ids):
    print(f" Document {index + 1} ID: {inserted_id}")


Successfully inserted 3 new documents!
Generated ObjectIDs:
 Document 1 ID: 6ad67e3324e6fe61eb2b30ee
 Document 2 ID: 6ad67e3324e6fe61eb2b30ef
 Document 3 ID: 6ad67e3324e6fe61eb2b30f0


## Adatok törlése (Delete)

Dokumentumok eltávolításához a `delete_one()` vagy `delete_many()` metódusokat használjuk. 
* Ha a törlésnek átadott szűrő üres (`{}`), a `delete_many()` a gyűjtemény **összes** elemét törli!

In [34]:

title = {"title": "MongoDB and Python"}

result = mycollection.delete_one(title)

print(f"Number of deleted docs: {result.deleted_count}")

# Ellenőrizzük, hogy maradt-e ilyen rekord
rest = mycollection.count_documents({"title": "MongoDB and Python"})
print(f"Number of remaining results: {rest}")


Number of deleted docs: 1
Number of remaining results: 0


## PyMongo CRUD Műveletek Összefoglalása

| Művelet (CRUD) | PyMongo metódus |
| :--- | :--- |
| **C**reate (Létrehozás) | `insert_one()`, `insert_many()` |
| **R**ead (Olvasás) | `find_one()`, `find()` |
| **U**pdate (Frissítés) | `update_one()`, `update_many()` |
| **D**elete (Törlés) | `delete_one()`, `delete_many()` |
